In [12]:
print("Jupyter está funcionando!")

Jupyter está funcionando!


In [13]:
# Cell 1: installs (run once)
# If you are on Colab you may need to restart the kernel after installs.

!pip -q install numpy pandas pillow tqdm faiss-cpu requests beautifulsoup4 wikipedia-api \
                 torch torchvision \
                 transformers accelerate \
                 google-generativeai


In [21]:
# Cell 2: imports + config

from __future__ import annotations

import os
import re
import json
import time
import random
import hashlib
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import faiss
import torch
from transformers import CLIPProcessor, CLIPModel

import requests
import wikipediaapi

GEMINI_AVAILABLE = False


@dataclass
class Config:
    # Local folders
    texts_dir: str = "./texts"
    images_dir: str = "./images"
    out_dir: str = "./out"

    # Text parsing
    section_code_regex: str = r"^\s*\[(?P<code>[A-Za-z0-9_:-]{3,})\]\s*"  # example: [ABC123] paragraph...
    code_must_be_at_start: bool = True

    # CLIP model
    clip_model_name: str = "openai/clip-vit-base-patch32"

    # Indexing
    paragraphs_parquet: str = "./out/paragraphs.parquet"
    faiss_index_path: str = "./out/clip_text.index"
    para_meta_path: str = "./out/para_meta.json"

    # Retrieval
    top_k_visual: int = 60
    top_k_web: int = 60
    top_k_judge: int = 30

    # Combine scores
    w_visual: float = 0.45
    w_web: float = 0.55

    # Web context (Wikipedia only for one-shot simplicity)
    wiki_lang: str = "en"
    web_cache_path: str = "./out/web_cache.json"

    # Rate limiting
    min_sleep_s: float = 0.15
    max_sleep_s: float = 0.6


cfg = Config()
os.makedirs(cfg.out_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Gemini available:", GEMINI_AVAILABLE)


Device: cpu
Gemini available: False


In [22]:
# Cell 3: helpers (parsing, IO)

code_re = re.compile(cfg.section_code_regex)

def stable_id(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:16]

def read_text_file(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def extract_code(paragraph: str) -> Optional[str]:
    if cfg.code_must_be_at_start:
        m = re.match(code_re, paragraph)
        return m.group("code") if m else None
    m = code_re.search(paragraph)
    return m.group("code") if m else None

def parse_image_filename(filename: str) -> Dict[str, Optional[str]]:
    base = os.path.splitext(os.path.basename(filename))[0]
    parts = base.split("_")

    out = {
        "raw": base,
        "type": None,
        "artist": None,
        "title": None,
        "date": None,
        "series_index": None
    }

    if len(parts) >= 2 and parts[0] in {"A", "P", "I"}:
        out["type"] = parts[0]
        tail = parts[1:]

        if tail and re.fullmatch(r"\(?\d+\)?", tail[-1]):
            out["series_index"] = re.sub(r"[()]", "", tail[-1])
            tail = tail[:-1]

        if tail and re.fullmatch(r"\d{4}|\d{4}-\d{2}-\d{2}", tail[-1]):
            out["date"] = tail[-1]
            tail = tail[:-1]

        if len(tail) >= 2:
            out["artist"] = tail[0].strip()
            out["title"] = "_".join(tail[1:]).strip()
        elif len(tail) == 1:
            out["artist"] = tail[0].strip()

    return out

def load_json(path: str) -> Dict:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_json(path: str, obj: Dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


In [23]:
import re
from typing import List, Dict, Optional

# Main header: E3.9.4. Title...
MAIN_HEADER_RE = re.compile(r"^(?P<main_code>E\d+(?:\.[0-9A-Za-z]+)*)\.\s+(?P<title>.+?)\s*$")

# Pure code line: E3.9.13.a.
PURE_CODE_RE = re.compile(r"^(?P<code>E\d+(?:\.[0-9A-Za-z]+)*)\.\s*$")

# Internal EL header line: EL3.87.A.1. Something...
# It also matches EL3.87. (with nothing after)
EL_HEADER_RE = re.compile(r"^(?P<el>EL\d+(?:\.[0-9A-Za-z]+)*)\.\s*(?P<rest>.*)$")

def _normalise(text: str) -> str:
    return text.replace("\r\n", "\n").replace("\r", "\n")

def is_entry_header(line: str) -> bool:
    s = line.strip()
    if not s.startswith("E"):
        return False
    if PURE_CODE_RE.match(s):
        return True
    if "_REF" in s or re.search(r"_RE\d+_", s):
        return True
    if ".wE" in s:
        return True
    return False

def extract_chunk_code(line: str) -> str:
    s = line.strip()

    m_pure = PURE_CODE_RE.match(s)
    if m_pure:
        return m_pure.group("code")

    # E3.9.4_REF2.2_LIT ... or E2.2.A.3.wE3.9.4.
    return s.split()[0].rstrip(".")

def is_el_header(line: str) -> Optional[Dict[str, str]]:
    s = line.strip()
    m = EL_HEADER_RE.match(s)
    if not m:
        return None
    return {"el_code": m.group("el").strip(), "rest": (m.group("rest") or "").strip()}

def parse_text_to_chunks(raw: str, min_chars: int = 30) -> List[Dict[str, str]]:
    raw = _normalise(raw)
    lines = raw.split("\n")

    chunks: List[Dict[str, str]] = []

    current_section_code: Optional[str] = None
    current_section_title: str = ""

    current_chunk_code: Optional[str] = None
    buffer: List[str] = []

    # Para split interno por EL
    current_inner_code: Optional[str] = None

    def flush():
        nonlocal buffer, current_inner_code
        if current_section_code is None:
            buffer = []
            current_inner_code = None
            return

        text = "\n".join(buffer).strip()
        if len(text) < min_chars:
            buffer = []
            current_inner_code = None
            return

        # chunk_code final: se tiver inner EL, vira "E4..._REF2_PSI|EL3.87.A.1"
        final_code = current_chunk_code or current_section_code
        if current_inner_code:
            final_code = f"{final_code}|{current_inner_code}"

        chunks.append({
            "section_code": current_section_code,
            "section_title": current_section_title,
            "chunk_code": final_code,
            "chunk_text": text
        })

        buffer = []
        current_inner_code = None

    for line in lines:
        s = line.strip()

        # 1) New main section
        m = MAIN_HEADER_RE.match(s)
        if m:
            flush()
            current_section_code = m.group("main_code")
            current_section_title = m.group("title")
            current_chunk_code = None
            buffer = []
            continue

        if current_section_code is None:
            continue

        # 2) New chunk entry inside section (E..._REF / .wE / pure code)
        if is_entry_header(s):
            flush()
            current_chunk_code = extract_chunk_code(s)
            current_inner_code = None
            buffer = [line]
            continue

        # 3) Split inside chunk when EL header appears
        el = is_el_header(s)
        if el:
            # Se já existe conteúdo no buffer, fecha o chunk anterior e abre outro
            if buffer and "\n".join(buffer).strip():
                flush()
            current_inner_code = el["el_code"]
            buffer = [line]
            continue

        # Split extra: um novo header REF no meio do chunk (evita juntar dois REFs num registro)
        is_ref_header_line = s.startswith("E") and ("_REF" in s or re.search(r"_RE\d+_", s))
        if buffer and is_ref_header_line:
            already_has_ref = any(
                (b.strip().startswith("E") and ("_REF" in b or re.search(r"_RE\d+_", b)))
                for b in buffer
            )
            if already_has_ref:
                flush()
                buffer = [line]
                continue

        buffer.append(line)

    flush()
    return chunks


In [29]:
#Cell 5: build do dataframe de chunks (chunks_df)

import os
import pandas as pd
import hashlib

def read_text_file(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def make_chunk_id(
    source_file: str,
    section_code: str,
    chunk_code: str,
    chunk_text: str) -> str:
    key = f"{source_file}||{section_code}||{chunk_code}||{chunk_text}"
    return hashlib.sha1(key.encode("utf-8")).hexdigest()


def build_chunks_df(texts_dir: str) -> pd.DataFrame:
    rows = []
    for root, _, files in os.walk(texts_dir):
        for fn in files:
            if not fn.lower().endswith((".txt", ".md")):
                continue
            path = os.path.join(root, fn)
            raw = read_text_file(path)

            chunks = parse_text_to_chunks(raw)
            for i, c in enumerate(chunks):
                chunk_id = make_chunk_id(
                    source_file=fn,
                    section_code=c["section_code"],
                    chunk_code=c["chunk_code"],
                    chunk_text=c["chunk_text"]
                )
            
                rows.append({
                    "chunk_id": chunk_id,          # <- NOVO ID ESTÁVEL
                    "source_file": fn,
                    "position": i,
                    "section_code": c["section_code"],
                    "section_title": c["section_title"],
                    "chunk_code": c["chunk_code"],
                    "chunk_text": c["chunk_text"],
                })


    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("Nenhum chunk encontrado. Confira texts_dir e o formato dos arquivos.")
    return df.reset_index(drop=True)

chunks_df = build_chunks_df(cfg.texts_dir)
print("Chunks:", len(chunks_df))
chunks_df.head(5)



Chunks: 5271


,chunk_id,source_file,position,section_code,section_title,chunk_code,chunk_text
0,147e6e60e66bdb641ca962c427f27ee35042459a,E.4. Can we be sure we are right_20122025 .txt,0,E4.1.A.1,"The Constructive Machine – Affect, Bias, and t...",E4.1.A.1,Biological and psychological mechanisms distor...
1,fe70e17927e8fdabd5e22cb866c59af4ed2780d8,E.4. Can we be sure we are right_20122025 .txt,1,E4.1.A.1,"The Constructive Machine – Affect, Bias, and t...",E4.1.A.1_REF1_SCI|EL3.102,"EL3.102. ""FEELINGS ACCOMPANY THE UNFOLDING OF ..."
2,bf67c9b0f627594576e014a5cd58e8e1c0871541,E.4. Can we be sure we are right_20122025 .txt,2,E4.1.A.1,"The Constructive Machine – Affect, Bias, and t...",E4.1.A.1_REF2_PSI|EL3.87,EL3.87. SYSTEM 1 AND SYSTEM 2\nDaniel Kahnema...
3,3388042bf55fe2ab5b68a0565086200afb084a44,E.4. Can we be sure we are right_20122025 .txt,3,E4.1.A.1,"The Constructive Machine – Affect, Bias, and t...",E4.1.A.1_REF2_PSI|EL3.87.A,EL3.87.A. WHAT YOU SEE IS ALL THERE IS (WYSIAT...
4,3fa7f01cf4b172fbfff3d323332d406062d158cf,E.4. Can we be sure we are right_20122025 .txt,4,E4.1.A.1,"The Constructive Machine – Affect, Bias, and t...",E4.1.A.1_REF2_PSI|EL3.87.A.1,"EL3.87.A.1. CAN WE THINK OUTSIDE THE BOX ? ""Yo..."


In [30]:
csv_path = os.path.join(cfg.out_dir, "chunks_full.csv")

chunks_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"  # bom para abrir no Excel sem quebrar acentos
)

print("CSV salvo em:", csv_path)


CSV salvo em: ./out\chunks_full.csv


In [31]:
# Cell 6: load CLIP

clip_model = CLIPModel.from_pretrained(cfg.clip_model_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(cfg.clip_model_name)

clip_model.eval()
print("Loaded:", cfg.clip_model_name)


Loaded: openai/clip-vit-base-patch32


In [32]:
# Cell 7: CLIP text embeddings for chunks + FAISS index

@torch.no_grad()
def clip_text_embeddings(texts: List[str], batch_size: int = 64, max_len: int = 77) -> np.ndarray:
    vecs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        # CLIP has a short context length. Long texts will be truncated by the tokenizer.
        inputs = clip_processor(
            text=batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        feats = clip_model.get_text_features(**inputs)
        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)  # normalise => IP == cosine
        vecs.append(feats.detach().cpu().numpy().astype(np.float32))
    return np.vstack(vecs)

# 1) texts -> embeddings
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_vecs = clip_text_embeddings(chunk_texts, batch_size=64)

# 2) build FAISS index (cosine similarity via normalised inner product)
dim = chunk_vecs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_vecs)

faiss.write_index(index, cfg.faiss_index_path)

# 3) save metadata (IMPORTANT: uses chunk_id)
chunk_meta = chunks_df[["chunk_id", "chunk_code", "chunk_text"]].to_dict("records")
save_json(cfg.para_meta_path, {"dim": dim, "chunk_meta": chunk_meta})

print("Saved index:", cfg.faiss_index_path)
print("Saved meta:", cfg.para_meta_path)
print("dim:", dim, "vectors:", len(chunk_meta))


100%|██████████| 83/83 [12:18<00:00,  8.90s/it]


Saved index: ./out/clip_text.index
Saved meta: ./out/para_meta.json
dim: 512 vectors: 5271


In [44]:
# Cell 8: CLIP image embeddings + retrieval (adapted for chunks_df + chunk_meta)

from PIL import Image, ImageFile

# Load index + meta
index = faiss.read_index(cfg.faiss_index_path)
meta_blob = load_json(cfg.para_meta_path)

# Ajuste aqui se o nome no JSON for diferente
# Esperado: {"dim": ..., "chunk_meta": [...]}
if "chunk_meta" in meta_blob:
    chunk_meta = meta_blob["chunk_meta"]
elif "para_meta" in meta_blob:
    # fallback caso você tenha salvado com nome antigo
    chunk_meta = meta_blob["para_meta"]
else:
    raise KeyError(f"Não encontrei 'chunk_meta' nem 'para_meta' em {cfg.para_meta_path}. Keys: {list(meta_blob.keys())}")

def get_chunk(pid: int) -> Dict[str, str]:
    """
    pid is the FAISS internal row id (0..N-1) returned by index.search
    """
    rec = chunk_meta[int(pid)]
    return {
        "chunk_id": rec.get("chunk_id", ""),
        "chunk_code": rec.get("chunk_code", ""),
        "chunk_text": rec.get("chunk_text", ""),
    }

# evita erro em imagens com arquivos parcialmente corrompidos
ImageFile.LOAD_TRUNCATED_IMAGES = True

# limite seguro para trabalhar (CLIP não precisa de imagem gigante)
MAX_SIDE = 2048  # pode ser 1024 se quiser ainda mais leve


@torch.no_grad()
def clip_image_embedding(image_path: str, max_side: int = MAX_SIDE) -> np.ndarray:
    try:
        img = Image.open(image_path)

        # tentativa de resize cedo
        w, h = img.size
        if max(w, h) > max_side:
            img = img.copy()
            img.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)

        img = img.convert("RGB")

        inputs = clip_processor(images=img, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        feats = clip_model.get_image_features(**inputs)
        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
        return feats.detach().cpu().numpy().astype(np.float32)  # shape [1, d]

    except Image.DecompressionBombError:
        # se ainda estourar, a forma mais segura é pular
        raise

@torch.no_grad()
def clip_text_embedding(text: str, max_len: int = 77) -> np.ndarray:
    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_len
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    feats = clip_model.get_text_features(**inputs)
    feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
    return feats.detach().cpu().numpy().astype(np.float32)  # shape [1, d]

def retrieve(query_vec: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    scores, ids = index.search(query_vec, k)
    return scores[0], ids[0]  # scores: (k,), ids: (k,)

def combine_candidates(
    visual_ids: np.ndarray, visual_scores: np.ndarray,
    web_ids: np.ndarray, web_scores: np.ndarray,
    w_visual: float, w_web: float,
    top_n: int
) -> List[Tuple[int, float]]:
    sv = {int(i): float(s) for i, s in zip(visual_ids, visual_scores) if int(i) >= 0}
    sw = {int(i): float(s) for i, s in zip(web_ids, web_scores) if int(i) >= 0}

    ids = set(sv.keys()) | set(sw.keys())
    merged = []
    for pid in ids:
        merged_score = (w_visual * sv.get(pid, 0.0)) + (w_web * sw.get(pid, 0.0))
        merged.append((pid, merged_score))

    merged.sort(key=lambda x: x[1], reverse=True)
    return merged[:top_n]


In [ ]:
# Cell 9: optional Gemini judge

def gemini_setup():
    if not GEMINI_AVAILABLE:
        raise RuntimeError("google-generativeai not installed.")
    api_key = os.environ.get(cfg.gemini_api_key_env)
    if not api_key:
        raise RuntimeError(f"Set env var {cfg.gemini_api_key_env} with your API key.")
    genai.configure(api_key=api_key)

def gemini_judge(image_path: str, meta: Dict, web_card: Dict, candidates: List[Dict]) -> Dict[str, object]:
    gemini_setup()
    model = genai.GenerativeModel(cfg.gemini_model)

    trimmed = []
    for c in candidates:
        t = c["section_text"]
        if len(t) > 900:
            t = t[:900] + "..."
        trimmed.append({"section_code": c["section_code"], "section_text": t, "score": c["score"]})

    prompt = f"""
You are curating pairings between an artwork image and philosophical paragraphs.

Use the image content and the web context card. If the web summary is empty, do not invent facts.
Choose the best section_code from the candidates.
Return strict JSON with:
best_section_code, confidence (0 to 1), rationale (1 to 2 sentences), runner_ups (up to 3 codes).

Filename meta: {meta}
Web context card: {web_card}
Candidates: {trimmed}
"""

    img = Image.open(image_path).convert("RGB")
    resp = model.generate_content([prompt, img])
    txt = (resp.text or "").strip()

    m = re.search(r"\{.*\}", txt, flags=re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass

    return {
        "best_section_code": trimmed[0]["section_code"] if trimmed else "",
        "confidence": 0.3,
        "rationale": txt[:1200],
        "runner_ups": [x["section_code"] for x in trimmed[1:4]]
    }


import wikipedia

wikipedia.set_lang("en")

def get_web_context(artist: str, title: str, date: str = "") -> dict:
    query = " ".join(x for x in [artist, title] if x)

    try:
        page = wikipedia.page(query, auto_suggest=True, redirect=True)
        return {
            "query": query,
            "page_title": page.title,
            "summary": page.summary[:1200],
            "notes": "source: wikipedia"
        }
    except Exception as e:
        return {
            "query": query,
            "page_title": "",
            "summary": "",
            "notes": f"wikipedia error: {type(e).__name__}"
        }


In [41]:
import os
from PIL import Image

# inclui formatos comuns + tif/tiff
IMG_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff", ".bmp")

suspects = []
errors = []

for root, _, files in os.walk(cfg.images_dir):
    for fn in files:
        if not fn.lower().endswith(IMG_EXTS):
            continue
        path = os.path.join(root, fn)
        rel = os.path.relpath(path, cfg.images_dir)

        try:
            with Image.open(path) as im:
                w, h = im.size
            px = w * h

            # limite mais baixo que o MAX do Pillow só para achar rápido
            if px > 150_000_000:
                suspects.append((rel, w, h, px))

        except Exception as e:
            errors.append((rel, type(e).__name__, str(e)))

suspects = sorted(suspects, key=lambda x: x[3], reverse=True)

print("Suspects (px > 150M):", len(suspects))
for s in suspects[:30]:
    print(s)

print("\nErrors while reading:", len(errors))
for e in errors[:30]:
    print(e)


Suspects (px > 150M): 0

Errors while reading: 2
('A_Georges Seurat_A Sunday on the island of La Grande Jatte_1884.jpg', 'DecompressionBombError', 'Image size (599100000 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.')
('A_Jean-Antoine Watteau_Embarquement for Cythere [Embarkation for Cythera]_1717.jpg', 'DecompressionBombError', 'Image size (258203946 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.')


In [47]:
# Cell 10: main loop (one best chunk per image)
# Output: results.jsonl with image_filename -> best_chunk_code (+ rationale)

import os
import json
import numpy as np
from tqdm import tqdm
from PIL import Image

results_path = os.path.join(cfg.out_dir, "results.jsonl")
os.makedirs(cfg.out_dir, exist_ok=True)

def load_done(path: str) -> set:
    done = set()
    if not os.path.exists(path):
        return done
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                fn = obj.get("image_filename", "")
                if fn:
                    done.add(fn)
            except Exception:
                continue
    return done

done = load_done(results_path)
print("Already done:", len(done))

# melhor: incluir mais formatos
IMG_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff", ".bmp")

# se não tem subpastas, os.listdir basta; se tiver, troque para os.walk
image_files = sorted([fn for fn in os.listdir(cfg.images_dir) if fn.lower().endswith(IMG_EXTS)])
print("Images found:", len(image_files))

# toggle simples para web (crie no cfg se quiser)
ENABLE_WEB = getattr(cfg, "enable_web", True) and (cfg.w_web > 0)

with open(results_path, "a", encoding="utf-8") as out_f:
    for fn in tqdm(image_files):
        if fn in done:
            continue

        image_path = os.path.join(cfg.images_dir, fn)
        meta = parse_image_filename(fn)

        # 1) Visual retrieval
        try:
            img_vec = clip_image_embedding(image_path)
        except Image.DecompressionBombError as e:
            obj = {
                "image_filename": fn,
                "meta": meta,
                "status": "image_error",
                "error_type": "DecompressionBombError",
                "error": str(e),
            }
            out_f.write(json.dumps(obj, ensure_ascii=False) + "\n")
            out_f.flush()
            continue
        except Exception as e:
            obj = {
                "image_filename": fn,
                "meta": meta,
                "status": "image_error",
                "error_type": type(e).__name__,
                "error": str(e),
            }
            out_f.write(json.dumps(obj, ensure_ascii=False) + "\n")
            out_f.flush()
            continue

        vis_scores, vis_ids = retrieve(img_vec, cfg.top_k_visual)

        # 2) Web retrieval (optional)
        web_card = {"query": "", "page_title": "", "summary": "", "notes": ""}
        web_scores, web_ids = np.array([], dtype=np.float32), np.array([], dtype=np.int64)

        if ENABLE_WEB and meta.get("artist") and meta.get("title"):
            try:
                web_card = get_web_context(meta.get("artist"), meta.get("title"), meta.get("date"))
            except Exception as e:
                web_card = {
                    "query": f'{meta.get("artist","")} {meta.get("title","")}'.strip(),
                    "page_title": "",
                    "summary": "",
                    "notes": f"get_web_context error: {type(e).__name__}: {e}"
                }

            web_text = " ".join([web_card.get("page_title", ""), web_card.get("summary", "")]).strip()
            if web_text and cfg.top_k_web > 0:
                try:
                    web_vec = clip_text_embedding(web_text)
                    web_scores, web_ids = retrieve(web_vec, cfg.top_k_web)
                except Exception as e:
                    web_card["notes"] = (web_card.get("notes","") + f" | web embedding error: {type(e).__name__}: {e}").strip()

        # 3) Combine
        merged = combine_candidates(
            vis_ids, vis_scores,
            web_ids, web_scores,
            cfg.w_visual, cfg.w_web,
            cfg.top_k_judge
        )

        candidates = []
        for pid, score in merged:
            c = get_chunk(int(pid))
            candidates.append({
                "chunk_id": c.get("chunk_id", ""),
                "chunk_code": c.get("chunk_code", ""),
                "chunk_text": c.get("chunk_text", ""),
                "score": float(score),
            })

        # 4) Decide best
        if not candidates:
            obj = {
                "image_filename": fn,
                "meta": meta,
                "web_card": web_card,
                "status": "no_candidates",
                "best_chunk_code": "",
                "confidence": 0.0,
                "rationale": "No candidates returned from FAISS search/merge.",
                "runner_ups": [],
                "top_candidates": [],
            }
            out_f.write(json.dumps(obj, ensure_ascii=False) + "\n")
            out_f.flush()
            continue

        if getattr(cfg, "use_gemini_judge", False):
            judge = gemini_judge(image_path, meta, web_card, candidates)
            best_code = judge.get("best_chunk_code", "") or judge.get("best_section_code", "")
            confidence = float(judge.get("confidence", 0.0) or 0.0)
            rationale = judge.get("rationale", "")
            runner_ups = judge.get("runner_ups", [])
        else:
            best_code = candidates[0]["chunk_code"]
            confidence = float(candidates[0]["score"])
            runner_ups = [c["chunk_code"] for c in candidates[1:4]]
            rationale = "Selected via weighted cosine similarity: CLIP(image,text) + CLIP(web,text)."

        obj = {
            "image_filename": fn,
            "meta": meta,
            "web_card": web_card,
            "status": "ok",
            "best_chunk_code": best_code,
            "confidence": confidence,
            "rationale": rationale,
            "runner_ups": runner_ups,
            "top_candidates": [{"chunk_code": c["chunk_code"], "score": c["score"]} for c in candidates[:10]],
        }

        out_f.write(json.dumps(obj, ensure_ascii=False) + "\n")
        out_f.flush()

print("Done. Saved:", results_path)


Already done: 1822
Images found: 2872


100%|██████████| 2872/2872 [17:53<00:00,  2.68it/s]  


Done. Saved: ./out\results.jsonl


In [49]:
# Cell 11: export final CSV (one row per image)

import json
import pandas as pd
import os

rows = []

with open(results_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        try:
            obj = json.loads(line)
        except Exception:
            continue

        meta = obj.get("meta") or {}

        rows.append({
            "image_filename": obj.get("image_filename", ""),
            "best_chunk_code": obj.get("best_chunk_code", ""),
            "confidence": obj.get("confidence", 0.0),
            "rationale": obj.get("rationale", ""),
            "artist": meta.get("artist", ""),
            "title": meta.get("title", ""),
            "date": meta.get("date", ""),
            "type": meta.get("type", ""),
        })

df = (
    pd.DataFrame(rows)
      .sort_values("confidence", ascending=False)
      .reset_index(drop=True)
)

final_csv = os.path.join(cfg.out_dir, "final_pairings.csv")
df.to_csv(final_csv, index=False, encoding="utf-8")

print("Wrote:", final_csv)
df.head(10)


Wrote: ./out\final_pairings.csv


,image_filename,best_chunk_code,confidence,rationale,artist,title,date,type
0,A_Gustav Klimt_Beethovenfries_1902_(3).jpg,E1.3.D.6_REF6.1_CUR,0.167399,Selected via weighted cosine similarity: CLIP(...,Gustav Klimt,Beethovenfries,1902,A
1,A_Gustav Klimt_Beethovenfries_1902_(9).jpg,E1.3.D.6_REF6.1_CUR,0.162499,Selected via weighted cosine similarity: CLIP(...,Gustav Klimt,Beethovenfries,1902,A
2,A_Alexander Calder_Southern cross_1963.PNG,E2.1.D.1.1.a,0.161288,Selected via weighted cosine similarity: CLIP(...,Alexander Calder,Southern cross,None,A
3,A_Édouard Vuillard_The Princess Bibesco_circa ...,E1.5.C.2.1.d_REF3_CUR,0.159919,Selected via weighted cosine similarity: CLIP(...,Édouard Vuillard,The Princess Bibesco_circa 1920,None,A
4,A_René Magritte_The Son Of Man_1964.jpg,E3.2.2_REF3_PO,0.159355,Selected via weighted cosine similarity: CLIP(...,René Magritte,The Son Of Man,None,A
5,A_Paula Rego_Dog women.PNG,E2.1.A.5.2.a.ii.2_REF1_CUR,0.158688,Selected via weighted cosine similarity: CLIP(...,Paula Rego,Dog women,None,A
6,A_Constantin Brancusi_Princess X_1915-1916.jpg,E3.4.4_REF7_CUR,0.158236,Selected via weighted cosine similarity: CLIP(...,Constantin Brancusi,Princess X_1915-1916,None,A
7,A_Giorgio Morandi_Natura Morta_1956.jpg,E1.6.E.2.6.b_REF2_CUR,0.156977,Selected via weighted cosine similarity: CLIP(...,Giorgio Morandi,Natura Morta,None,A
8,A_Lidia Lisboa_from the Series umbilical cord_...,E2.1.D.1.1.a,0.156903,Selected via weighted cosine similarity: CLIP(...,Lidia Lisboa,from the Series umbilical cord,2024,A
9,A_Liliana Porter_1973_(2).jpg,E2.1.D.1.1.a,0.155966,Selected via weighted cosine similarity: CLIP(...,Liliana Porter,None,1973,A


In [ ]:
# Cell 12: optional: enable Gemini judge

# 1) Set your key for this notebook session:
# os.environ["GEMINI_API_KEY"] = "PASTE_KEY_HERE"

# 2) Turn on the judge:
# cfg.use_gemini_judge = True
# cfg.gemini_model = "gemini-1.5-flash"  # or the exact model you used

print("use_gemini_judge:", cfg.use_gemini_judge)


In [ ]:
# If your paragraph codes are not like [CODE] at the start, update:
# cfg.section_code_regex
# cfg.code_must_be_at_start

# CLIP truncates long paragraphs. If your paragraphs are long, consider pre-shortening:
# sections_df["section_text"] = sections_df["section_text"].str.slice(0, 800)
# before embedding, or switch to a text embedding model for paragraphs and keep CLIP only for image queries.
